# Parking Lot Vehicle Monitoring with YOLO

# Objective

To design and implement a computer vision-based system capable of monitoring parking spaces using video, detecting when each space is occupied or vacated, and recognizing the visual characteristics of the vehicles involved to identify changes in the space's occupancy.

# Considerations

- Fixed camera.
- Just track color.
- For our projects there is a small sample of camera data in `/kaggle/input/datasets/tomasreisenman/reference/reference/`.

# Models Used

- **YOLOv8m**: vehicle detection.
- **OpenCLIP**: car presence (recognizing that the same car is still there across captures).
- **K-Means**: color detection.

# Dataset

- YOLO is pretrained on the **COCO** dataset. Here we fine-tuned it on [Vehicles-OpenImages](https://universe.roboflow.com/roboflow-gw7yv/vehicles-openimages) (Roboflow), classes: `Car`, `Motorcycle`, `Bus`, `Truck`, `Ambulance`.
- CLIP is pretrained and we do **not** fine-tune it.

# Pipeline

1. Fine-tune the YOLO detector on the vehicle dataset.
2. Detect whether each hand-calibrated parking spot is occupied or empty.
3. Track cars across captures to determine when a car arrives, stays or leaves.
4. Extract each detected car's dominant color.

Everything runs on the GPU by default (GPU `0` on Kaggle's T4 x2); one boolean flag in Section 3 switches the whole notebook to CPU.


## 1. Install dependencies

We need:
- `ultralytics` — YOLO training/inference (YOLOv8).
- `roboflow` — to download the training dataset in YOLO format.
- `shapely` — polygon geometry for parking-spot occupancy checks.
- `opencv-python-headless` — image loading/drawing.
- `scikit-learn` — K-Means for dominant-color extraction.
- `open_clip_torch` — pretrained CLIP model, used **zero-shot** (no training) for brand/model guessing in the final module.

Kaggle already has `torch`, `numpy`, `matplotlib`, and `Pillow` preinstalled, so we don't reinstall those.

In [ ]:
!pip install -q ultralytics roboflow shapely opencv-python-headless scikit-learn open_clip_torch pandas

## 2. Imports

Standard imports for the whole notebook. Nothing here trains or downloads anything yet.

In [ ]:
import os
import json
import glob
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from shapely.geometry import Polygon, box as shapely_box

import torch
from ultralytics import YOLO

print("Setup OK. Libraries imported.")

## 3. Choose the device (GPU / CPU)

One boolean controls the device for the **entire** notebook — every training and inference cell below reads `FORCE_DEVICE` from this cell:

- **`USE_CPU = False` (default)** → uses the GPU, device `0` (the first GPU of Kaggle's T4 x2).
- **`USE_CPU = True`** → forces `"cpu"` everywhere.

Flip that one flag and nothing else in the notebook needs to change.

In [ ]:
# Single switch for the whole notebook:
#   False -> GPU (device 0 — the first GPU of Kaggle's T4 x2)
#   True  -> CPU
USE_CPU = False

if USE_CPU:
    FORCE_DEVICE = "cpu"
else:
    FORCE_DEVICE = 0  # first GPU (T4 x2 on Kaggle)

# Model selection — change this single variable to switch models
MODEL_NAME = "yolov8m.pt"  # e.g. "yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov11n.pt", etc.

print(f"FORCE_DEVICE set to: {FORCE_DEVICE!r}")
print(f"MODEL_NAME: {MODEL_NAME}")
if not USE_CPU and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  - cuda:{i} -> {torch.cuda.get_device_name(i)}")

## 4. Get the training dataset

We use **`Vehicles-OpenImages`**, a public dataset on Roboflow Universe with the classes: `Ambulance`, `Bus`, `Car`, `Motorcycle`, `Truck`. This gives us both `Car` and `Motorcycle` — the two classes our parking lot needs.

Two versions are available as Kaggle inputs:

- **`vehicles-openimages`** — standard YOLO format (xywh labels), used by the default `yolov8` variant
- **`vehicles-openimages-obb`** — OBB format (4-corner rotated labels), used by the `yolo26_obb` variant

The active dataset is selected by `MODEL_VARIANT` in Section 3.

**Make sure you've added the zip as a Kaggle Input dataset (see the steps above) before running this.**

In [ ]:
# Dataset path (explicit — no glob searching)
DATASET_DIR = "/kaggle/input/datasets/tomasreisenman/vehicles-openimages"
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print(f"Using dataset: {DATASET_DIR}")
print(f"Using model: {MODEL_NAME}")

## 5. Inspect the dataset structure

Before training, let's sanity-check what we actually downloaded: the `data.yaml` (class names + folder layout) and how many images/labels exist per split.

In [ ]:
# data_yaml_path is already set in Section 4
with open(data_yaml_path, "r") as f:
    print(f.read())

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    if os.path.isdir(img_dir):
        n_images = len(glob.glob(os.path.join(img_dir, "*")))
        print(f"{split}: {n_images} images")
    else:
        print(f"{split}: folder not found")

## 6. Visual sanity check of the training data

Let's plot a handful of training images with their ground-truth bounding boxes drawn, just to confirm the labels look correct before we spend CPU time training on them.

In [ ]:
def yolo_txt_to_boxes(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((int(cls), x1, y1, x2, y2))
    return boxes


train_img_dir = os.path.join(DATASET_DIR, "train", "images")
train_lbl_dir = os.path.join(DATASET_DIR, "train", "labels")
sample_imgs = glob.glob(os.path.join(train_img_dir, "*"))[:4]

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(20, 5))
for ax, img_path in zip(axes, sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = os.path.join(train_lbl_dir, Path(img_path).stem + ".txt")
    for cls, x1, y1, x2, y2 in yolo_txt_to_boxes(label_path, w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Train YOLO

Training choices, and why — tuned for the default GPU runtime (T4 x2); flip the boolean in Section 3 to switch to CPU:

- **Model: `yolov8m.pt`** (medium) — current choice, see the comparison in Section 7b. It won on final validation mAP50 among the models tested. On CPU it is noticeably slower per epoch than nano/small, but on GPU (like your T4 x2) it trains in minutes — change `MODEL_NAME` in Section 3 to try another variant.
- **Image size: 416** instead of the default 640 — smaller images mean roughly 2-3x faster CPU inference/training with a moderate accuracy trade-off. On GPU this constraint matters less, but it's left as-is to keep results comparable across runs.
- **Epochs: 30** — enough to see the loss curves converge without an excessive run time on GPU.
- **Batch size: auto on GPU, 8 on CPU** — `batch=-1` tells `ultralytics` to automatically pick the largest batch size that fits in GPU memory, which uses your T4s far more efficiently than a fixed small batch. On CPU there's no such benefit, so it stays fixed at 8 to keep memory use predictable.

On CPU, expect this to take a while (potentially 30+ minutes). On GPU it should be a few minutes. Ultralytics prints per-epoch progress so you can monitor it either way.

In [ ]:
model = YOLO(MODEL_NAME)  # starts from COCO-pretrained weights, then fine-tunes on our dataset

train_batch = -1 if FORCE_DEVICE != "cpu" else 8  # auto-batch on GPU, fixed small batch on CPU

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=416,
    batch=train_batch,
    device=FORCE_DEVICE,   # from Section 3 — 0 (first GPU) by default, "cpu" if USE_CPU
    workers=2,
    # --- Data augmentation ---
    hsv_h=0.015,      # slight hue jitter
    hsv_s=0.7,        # saturation jitter — helps with lighting/paint-color variety
    hsv_v=0.4,        # brightness jitter — useful given your day/night frames
    degrees=15.0,     # moderate rotation
    translate=0.1,    # small positional jitter
    scale=0.5,        # scale jitter — helps with camera-distance/zoom mismatch
    shear=2.0,        # mild shear — approximates minor perspective variation
    perspective=0.0002,
    flipud=0.0,       # vertical flip — disabled: camera is oblique, not top-down
    fliplr=0.5,       # horizontal flip, standard
    mosaic=1.0,       # combine 4 images per batch
    mixup=0.1,        # light regularization
    erasing=0.4,      # random erasing
    project="parking_yolo",
    name="train_run",
    patience=10,           # stop early if validation stops improving
)

BEST_WEIGHTS = os.path.join(results.save_dir, "weights", "best.pt")
print(f"Best weights saved at: {BEST_WEIGHTS}")

## 7b. Training results comparison — YOLOv8 / YOLO11 / YOLO26 (n, s, m)

Snapshot of the runs logged in `../results.md` (all at `imgsz=416` on the Vehicles-OpenImages validation split: 250 images, 454 instances).

**Winner: `yolov8m` ✅** — highest validation mAP50, hence `MODEL_NAME = "yolov8m.pt"` in Section 3.

### What the metrics mean

- **mAP50** — mean Average Precision at IoU ≥ 0.5, averaged over all classes. The headline detection-quality number: higher is better.
- **mAP50-95** — same, but averaged over IoU thresholds 0.5–0.95. Stricter; rewards tight boxes, not just roughly-correct ones.
- **P (precision)** — of all boxes the model predicted, the fraction that was correct. Low precision = many false alarms.
- **R (recall)** — of all real objects, the fraction the model found. Low recall = many missed cars.
- **box_loss** — error in predicted box coordinates. Lower is better, but only comparable between runs of the same setup.
- **cls_loss** — error in predicted class labels (Car vs Bus vs ...). Lower is better.
- **dfl_loss** — distribution focal loss, fine-grained quality of box edges. Lower is better.
- **Params / GFLOPs** — model size and compute cost. Bigger models score higher but train/run slower.

### Results

| Model | Params | GFLOPs | box_loss | cls_loss | dfl_loss | P | R | mAP50 | mAP50-95 |
|---|---|---|---|---|---|---|---|---|---|
| yolov8n | 3.2M | 8.7 | 0.695 | 0.690 | 1.030 | 0.595 | 0.552 | 0.582 | 0.457 |
| yolov8s | 11.2M | 28.6 | 0.750 | 0.680 | 1.100 | 0.610 | 0.580 | 0.595 | 0.440 |
| **yolov8m ✅** | **25.8M** | **78.7** | 0.809 | 0.613 | 1.171 | 0.650 | 0.563 | **0.607** | 0.430 |
| yolo11n | 2.6M | 6.5 | 0.604 | 0.448 | 0.981 | 0.590 | 0.642 | 0.598 | 0.460 |
| yolo11s | 9.4M | 21.5 | 0.980 | 0.990 | 1.300 | 0.600 | 0.590 | 0.586 | 0.405 |
| yolo11m | 20.0M | 67.8 | 0.947 | 0.926 | 1.270 | 0.575 | 0.600 | 0.581 | 0.412 |
| yolo26n | 2.9M | 7.2 | 0.660 | 0.550 | 1.020 | 0.570 | 0.600 | 0.570 | 0.420 |
| yolo26s | 10.5M | 25.0 | 0.800 | 0.700 | 1.150 | 0.600 | 0.590 | 0.590 | 0.425 |
| yolo26m | 22.0M | 72.0 | 0.850 | 0.680 | 1.190 | 0.630 | 0.580 | 0.602 | 0.428 |

## 8. Validate the trained model

Run the built-in validation to get standard detection metrics (precision, recall, mAP@0.5, mAP@0.5:0.95) on the held-out validation split. These numbers tell us how much to trust the detector before relying on it for occupancy/tracking logic.

In [ ]:
trained_model = YOLO(BEST_WEIGHTS)
metrics = trained_model.val(data=data_yaml_path, imgsz=416, device=FORCE_DEVICE)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 9. Quick inference sanity check

Before moving to parking-lot-specific logic, let's confirm the trained model draws sensible boxes on a few validation images.

In [ ]:
val_img_dir = os.path.join(DATASET_DIR, "valid", "images")
test_samples = glob.glob(os.path.join(val_img_dir, "*"))[:4]

pred_results = trained_model.predict(test_samples, imgsz=416, device=FORCE_DEVICE, conf=0.25)

fig, axes = plt.subplots(1, len(pred_results), figsize=(20, 5))
for ax, r in zip(axes, pred_results):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9b. Rotation alignment check — vertical vs horizontal vs original

The longest structural line in the reference photo (spot edge, curb) sets the
parking-lot direction. From its angle we build **two** rotations:

- **vertical (current)** — rotates by `90 - angle`, so the longest line ends up vertical.
- **horizontal (previous)** — rotates by `-angle`, so the longest line ends up horizontal.
- **original** — no rotation at all.

All three variants of every photo in `reference/` are run through the trained
model, so we can see which orientation YOLO detects best.

In [ ]:
# --- 9b setup: line detection + both rotation matrices ---
# Reference photos live here (same folder Section 12 uses later as BASE_DIR)
REF_DIR = "/kaggle/input/datasets/tomasreisenman/reference/reference/"
REFERENCE_IMAGE_FOR_HOMOGRAPHY = REF_DIR + "reference_image.jpg"

import glob as _glob
REF_IMAGES = sorted(_glob.glob(os.path.join(REF_DIR, "*.jpg")))
print(f"Found {len(REF_IMAGES)} reference photos in {REF_DIR}")
for p in REF_IMAGES:
    print(f"  {os.path.basename(p)}")


def preprocess_for_lines(image_bgr, blur_kernel=(5, 5)):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(gray, blur_kernel, 0)


def line_angle_deg(x1, y1, x2, y2):
    return np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180  # direction only, 0-180


def filter_border_lines(lines, image_shape, border_threshold=10, angle_threshold=10):
    """Discard lines nearly parallel to an image border AND close to it.

    The Hough transform often detects the image border itself as a line,
    which is not a useful structural line.
    """
    if len(lines) == 0:
        return lines
    h, w = image_shape[:2]
    filtered = []
    for line in lines:
        x1, y1, x2, y2 = line
        angle = line_angle_deg(x1, y1, x2, y2)
        is_horizontal = angle < angle_threshold or angle > (180 - angle_threshold)
        if is_horizontal:
            if min(y1, y2) < border_threshold or max(y1, y2) > (h - border_threshold):
                continue
        is_vertical = (90 - angle_threshold) < angle < (90 + angle_threshold)
        if is_vertical:
            if min(x1, x2) < border_threshold or max(x1, x2) > (w - border_threshold):
                continue
        filtered.append(line)
    return np.array(filtered, dtype=int) if filtered else np.empty((0, 4), dtype=int)


def detect_line_segments(image_bgr, canny_low=50, canny_high=150,
                          hough_threshold=60, min_line_length=60, max_line_gap=15):
    gray = preprocess_for_lines(image_bgr)
    edges = cv2.Canny(gray, canny_low, canny_high)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=hough_threshold,
                             minLineLength=min_line_length, maxLineGap=max_line_gap)
    if lines is None:
        return np.empty((0, 4), dtype=int)
    lines = lines.reshape(-1, 4)
    return filter_border_lines(lines, image_bgr.shape, border_threshold=10, angle_threshold=10)


def line_length(line):
    x1, y1, x2, y2 = line
    return np.hypot(x2 - x1, y2 - y1)


def longest_line(lines):
    """Return the single longest line from the array."""
    if len(lines) == 0:
        return None
    lengths = np.array([line_length(l) for l in lines])
    return lines[np.argmax(lengths)]


def rotation_matrices_for_angle(angle_deg, image_shape):
    """Build both rotations from the longest-line angle.

    Returns (M_vertical, M_horizontal): 2x3 matrices for cv2.warpAffine.
    """
    h, w = image_shape[:2]
    center = (w // 2, h // 2)
    M_vertical = cv2.getRotationMatrix2D(center, 90 - angle_deg, 1.0)    # current
    M_horizontal = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)     # previous
    return M_vertical, M_horizontal


def apply_rotation(image_bgr, M):
    h, w = image_bgr.shape[:2]
    return cv2.warpAffine(image_bgr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


# --- Calibrate once on the clearest reference photo ---
ref_img = cv2.imread(REFERENCE_IMAGE_FOR_HOMOGRAPHY)
if ref_img is None:
    raise FileNotFoundError(f"Could not read {REFERENCE_IMAGE_FOR_HOMOGRAPHY}")

detected_lines = detect_line_segments(ref_img)
LONGEST = longest_line(detected_lines)
if LONGEST is None:
    raise RuntimeError("No lines detected in the reference image — try adjusting Hough params.")

LONGEST_ANGLE = line_angle_deg(*LONGEST)
if LONGEST_ANGLE >= 90:
    LONGEST_ANGLE -= 180
print(f"Longest line: {LONGEST.tolist()}  angle: {LONGEST_ANGLE:.1f} deg")

ROTATION_VERTICAL, ROTATION_HORIZONTAL = rotation_matrices_for_angle(LONGEST_ANGLE, ref_img.shape)

# Debug view: what the line detector sees
preprocessed = preprocess_for_lines(ref_img)
edges_vis = cv2.Canny(preprocessed, 50, 150)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Reference (original)")
axes[0].axis("off")
axes[1].imshow(preprocessed, cmap="gray")
axes[1].set_title("Blurred greyscale (line-detection input)")
axes[1].axis("off")
axes[2].imshow(edges_vis, cmap="gray")
axes[2].set_title("Canny edges")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# Longest line overlay + both rotations of the reference
lines_vis = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB).copy()
x1, y1, x2, y2 = [int(v) for v in LONGEST]
cv2.line(lines_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(lines_vis)
axes[0].set_title(f"Longest line (red, {LONGEST_ANGLE:.1f} deg)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(apply_rotation(ref_img, ROTATION_VERTICAL), cv2.COLOR_BGR2RGB))
axes[1].set_title("Vertical rotation (current)")
axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(apply_rotation(ref_img, ROTATION_HORIZONTAL), cv2.COLOR_BGR2RGB))
axes[2].set_title("Horizontal rotation (previous)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- 9b comparison: original vs vertical vs horizontal on all reference photos ---
def predict_boxes(image_bgr, model, imgsz=416, conf=0.50):
    res = model.predict(image_bgr, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False, iou=0.4)[0]
    boxes = []
    if res.boxes is not None:
        for b in res.boxes:
            boxes.append((res.names[int(b.cls)], float(b.conf), [round(v, 1) for v in b.xyxy[0].tolist()]))
    return res, boxes


combo_rows = []
for img_path in REF_IMAGES:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"Skipping unreadable: {img_path}")
        continue
    variants = {
        "original": img_bgr,
        "vertical": apply_rotation(img_bgr, ROTATION_VERTICAL),
        "horizontal": apply_rotation(img_bgr, ROTATION_HORIZONTAL),
    }
    print(f"\n=== {os.path.basename(img_path)} ===")
    plotted = {}
    for name, variant in variants.items():
        res, boxes = predict_boxes(variant, trained_model)
        confs = [c for _, c, _ in boxes]
        combo_rows.append({
            "image": os.path.basename(img_path),
            "variant": name,
            "n_detections": len(boxes),
            "mean_conf": round(float(np.mean(confs)), 3) if confs else 0.0,
        })
        print(f"  {name}: {len(boxes)} detections" + (f"  mean_conf={np.mean(confs):.3f}" if confs else ""))
        for cls_name, cf, bx in boxes:
            print(f"    {cls_name}  conf={cf:.3f}  box={bx}")
        plotted[name] = res.plot()

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    for ax, name in zip(axes, ["original", "vertical", "horizontal"]):
        ax.imshow(cv2.cvtColor(plotted[name], cv2.COLOR_BGR2RGB))
        ax.set_title(name)
        ax.axis("off")
    plt.suptitle(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()

combo_df = pd.DataFrame(combo_rows)
print(combo_df.pivot_table(index="image", columns="variant", values="n_detections", aggfunc="first"))
print(combo_df.pivot_table(index="image", columns="variant", values="mean_conf", aggfunc="first"))

# Static Spots

Using static spots for parking occupation
## 10. Auto-calibrate parking spots from a reference image

Instead of hard-coding polygon coordinates, this section uses a **reference image with cars already parked** and the trained model to automatically extract the parking spots.

**How it works:**
1. Load the reference image (cars parked in their spots)
2. Run the trained model to detect all vehicles
3. For each detected vehicle, create a parking spot polygon around it (bounding box expanded by `SPOT_EXPANSION` to approximate the real spot boundaries)
4. Visualize the extracted spots for verification

**One-time per camera position** — if the camera moves, re-run this section with a new reference image.

In [ ]:
# --- Auto-calibrate parking spots from a reference image with cars ---
CALIBRATION_IMAGE_PATH = "/kaggle/input/datasets/tomasreisenman/reference/reference/20260912171801.jpg"

# Expansion factor: cars are smaller than spots, so we expand each detection
# bounding box by this fraction on each side to approximate the spot boundaries.
# Increase if spots are too tight, decrease if they overlap too much.
SPOT_EXPANSION = 0.0

# Load calibration image and run detection
calib_img_bgr = cv2.imread(CALIBRATION_IMAGE_PATH)
if calib_img_bgr is None:
    raise FileNotFoundError(f"Could not read {CALIBRATION_IMAGE_PATH}")

calib_result = trained_model.predict(CALIBRATION_IMAGE_PATH, imgsz=416, device=FORCE_DEVICE, conf=0.50, verbose=False,iou=0.4)[0]

# Extract parking spots from detections
PARKING_SPOTS = {}
spot_idx = 1
for b in calib_result.boxes:
    cls_name = calib_result.names[int(b.cls)]
    if cls_name not in ("Car", "Motorcycle"):
        continue
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    w = x2 - x1
    h = y2 - y1
    # Expand bounding box to approximate spot size
    x1_exp = max(0, int(x1 - w * SPOT_EXPANSION))
    y1_exp = max(0, int(y1 - h * SPOT_EXPANSION))
    x2_exp = min(calib_img_bgr.shape[1], int(x2 + w * SPOT_EXPANSION))
    y2_exp = min(calib_img_bgr.shape[0], int(y2 + h * SPOT_EXPANSION))
    # Create polygon (rectangle in image coordinates)
    polygon = [(x1_exp, y1_exp), (x2_exp, y1_exp), (x2_exp, y2_exp), (x1_exp, y2_exp)]
    vehicle_type = "motorcycle" if cls_name == "Motorcycle" else "car"
    PARKING_SPOTS[f"spot_{spot_idx}_{vehicle_type}"] = {
        "polygon": polygon,
        "vehicle_type": vehicle_type
    }
    spot_idx += 1

print(f"Extracted {len(PARKING_SPOTS)} parking spots from calibration image:")
for name, spot in PARKING_SPOTS.items():
    print(f"  {name}: {spot['vehicle_type']}  polygon={spot['polygon']}")

# Visualize the extracted spots
def draw_calibration(image_path, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    overlay = img.copy()
    for name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        color = (0, 255, 255) if spot["vehicle_type"] == "car" else (255, 0, 255)
        cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=3)
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(overlay, name, (cx - 40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("Auto-extracted parking spots — do these align with the real spots?")
    plt.show()

draw_calibration(CALIBRATION_IMAGE_PATH, PARKING_SPOTS)

## 11. Per-spot occupancy detection

For each calibrated spot, we check whether any detected vehicle box of the **matching class** overlaps it enough to count as "occupied":

- `vehicle_type == "car"` → only `Car` detections are considered.
- `vehicle_type == "motorcycle"` → only `Motorcycle` detections are considered (as agreed, this spot never checks for cars, brand, or model).

Overlap is measured as **intersection area / spot polygon area** using `shapely` (not a plain box IoU, since spots are irregular quadrilaterals). A spot is "occupied" if this ratio passes a threshold (default 30% — tune this once you see it running on real photos).

In [ ]:
OCCUPANCY_THRESHOLD = 0.80

def check_occupancy(image_path, model, spots, conf=0.25, imgsz=416):
    result = model.predict(image_path, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    names = result.names  # class id -> class name

    detections = []
    for b in result.boxes:
        cls_name = names[int(b.cls)]
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        detections.append({"class": cls_name, "box": (x1, y1, x2, y2), "conf": float(b.conf)})

    occupancy = {}
    for spot_name, spot in spots.items():
        spot_poly = Polygon(spot["polygon"])
        expected_class = "Car" if spot["vehicle_type"] == "car" else "Motorcycle"

        occupied = False
        matched_box = None
        for det in detections:
            if det["class"] != expected_class:
                continue
            det_poly = shapely_box(*det["box"])
            if not spot_poly.intersects(det_poly):
                continue
            overlap_ratio = spot_poly.intersection(det_poly).area / spot_poly.area
            if overlap_ratio >= OCCUPANCY_THRESHOLD:
                occupied = True
                matched_box = det["box"]
                break

        occupancy[spot_name] = {"occupied": occupied, "box": matched_box}

    return occupancy, detections

## 12. Run occupancy check on a single frame and visualize it

Green outline = occupied, red outline = empty — a quick visual confirmation that matches what a human glancing at the photo would say.

The cell also verifies that `TEST_FRAME_PATH` and every image in `TEST_IMAGES` (used by the two debug cells below) actually exists under `BASE_DIR`.

In [ ]:
def visualize_occupancy(image_path, occupancy, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    for spot_name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        is_occupied = occupancy[spot_name]["occupied"]
        color = (0, 200, 0) if is_occupied else (220, 0, 0)
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=3)
        label = f"{spot_name}: {'OCCUPIED' if is_occupied else 'EMPTY'}"
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(img, label, (cx - 60, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

BASE_DIR = "/kaggle/input/datasets/tomasreisenman/reference/reference/"

TEST_FRAME_PATH = BASE_DIR + "20260915212238" + ".jpg"

TEST_IMAGES = [
    BASE_DIR + "20260910200600.jpg",
    BASE_DIR + "20260911172532.jpg",
    BASE_DIR + "20260911194005.jpg",
    BASE_DIR + "20260912171801.jpg",
    BASE_DIR + "20260912191134.jpg",
    BASE_DIR + "20260913005125.jpg",
    BASE_DIR + "20260914191018.jpg",
    BASE_DIR + "20260915212238.jpg",
    BASE_DIR + "20260916110642.jpg",
    BASE_DIR + "20260920190053.jpg",
    BASE_DIR + "reference_image.jpg",
]

missing = [p for p in [TEST_FRAME_PATH] + TEST_IMAGES if not os.path.exists(p)]
if missing:
    print("Warning — not found (check the dataset was added and BASE_DIR is correct):")
    for p in missing:
        print(f"  {p}")
else:
    print(f"All {len(TEST_IMAGES)} test images found.")

if os.path.exists(TEST_FRAME_PATH):
    occupancy, detections = check_occupancy(TEST_FRAME_PATH, trained_model, PARKING_SPOTS)
    print(occupancy)
    visualize_occupancy(TEST_FRAME_PATH, occupancy, PARKING_SPOTS)
else:
    print("TEST_FRAME_PATH not found — skipping the occupancy check.")

In [ ]:
# See every raw detection with its confidence, before spot-matching logic touches it
raw_results = {}
for img_path in TEST_IMAGES:
    if not os.path.exists(img_path):
        print(f"Skipping missing: {img_path}")
        continue
    result = trained_model.predict(img_path, imgsz=416, device=FORCE_DEVICE, conf=0.5, verbose=False, iou=0.4)[0]
    raw_results[img_path] = result
    print(f"\n{os.path.basename(img_path)}")
    for b in result.boxes:
        box = [round(v, 1) for v in b.xyxy[0].tolist()]
        print(f"  {result.names[int(b.cls)]}  conf={float(b.conf):.3f}  box={box}")

In [ ]:
# Visualize the raw boxes directly on the image
import matplotlib.pyplot as plt

paths = [p for p in TEST_IMAGES if p in raw_results]
cols = 4
rows = (len(paths) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
for ax, img_path in zip(axes.flat, paths):
    ax.imshow(cv2.cvtColor(raw_results[img_path].plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis("off")
for ax in axes.flat[len(paths):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import glob

counts = Counter()
for label_file in glob.glob(f"{DATASET_DIR}/train/labels/*.txt"):
    with open(label_file) as f:
        for line in f:
            cls_id = int(line.split()[0])
            counts[cls_id] += 1

print(counts)  # compare count for class id 3 (Motorcycle) vs class id 2 (Car)

## 13. Car color from YOLO detections

For each Car that YOLO detects in the test frame, we estimate its color in four steps:

Crop the detection box and trim CROP_SHRINK from each side, so most background (asphalt, neighbouring cars) is excluded.
Cluster the crop's pixels with K-Means (N_CLUSTERS = 4) and keep the largest non-dark cluster (at least MIN_BODY_SHARE of the crop), so shadows, glass and tires don't win. Dim frames are first scaled up (brighten_frame), which keeps hue and saturation.
Name it by converting the dominant RGB to HSV and applying simple rules: value/saturation decide black, white, silver and gray; hue decides the chromatic colors.
Show the annotated frame, the exact crops that were analysed, and a swatch of each dominant color.

Test frame: 20260915212238.jpg (from BASE_DIR, Section 12). It uses trained_model, FORCE_DEVICE and BASE_DIR from earlier sections, so run those first.

Limitations: this frame was taken at night, so artificial lighting shifts the colors (white can read as yellow/orange). The all_clusters column lists every cluster found per car, to help tune DARK_V, MIN_BODY_SHARE and CROP_SHRINK. If the printed saturation warning appears, the camera is in IR/grayscale mode and no method can recover color.

In [ ]:
# --- 13. Car color from YOLO detections ---
# Requires from earlier sections: trained_model, FORCE_DEVICE, BASE_DIR
from sklearn.cluster import KMeans

COLOR_TEST_PATH = BASE_DIR + "20260915212238.jpg"
COLOR_CONF = 0.25
COLOR_CLASSES = ("Car",)   # add "Motorcycle" to include it
CROP_SHRINK = 0.2          # fraction trimmed from each side of the box
N_CLUSTERS = 4
DARK_V = 60             # clusters with max(R,G,B) below this = shadow / glass / tires
MIN_BODY_SHARE = 0.15   # a non-dark cluster needs >= this share to beat dark ones
BRIGHT_TARGET = 230     # dim frames are scaled so their 99th-percentile brightness hits this


def brighten_frame(img_bgr, target=BRIGHT_TARGET):
    """Scale a dim (night) frame up; hue and saturation are preserved."""
    v = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[..., 2]
    p99 = float(np.percentile(v, 99))
    if p99 >= target or p99 < 1:
        return img_bgr
    return np.clip(img_bgr.astype(np.float32) * (target / p99), 0, 255).astype(np.uint8)


def crop_box_center(img_bgr, box, shrink=CROP_SHRINK):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    h_img, w_img = img_bgr.shape[:2]
    x1 = int(max(0, x1 + w * shrink)); x2 = int(min(w_img, x2 - w * shrink))
    y1 = int(max(0, y1 + h * shrink)); y2 = int(min(h_img, y2 - h * shrink))
    return img_bgr[y1:y2, x1:x2]


def color_clusters(crop_bgr, k=N_CLUSTERS):
    """K-Means on the crop's pixels -> [(rgb, share), ...] sorted by share, largest first."""
    scale = 64 / max(crop_bgr.shape[:2])
    if scale < 1:
        crop_bgr = cv2.resize(crop_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    pixels = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).reshape(-1, 3).astype(np.float32)
    km = KMeans(n_clusters=min(k, len(pixels)), n_init=10, random_state=0).fit(pixels)
    counts = np.bincount(km.labels_, minlength=len(km.cluster_centers_))
    order = np.argsort(-counts)
    return [(tuple(int(c) for c in km.cluster_centers_[i]), counts[i] / counts.sum()) for i in order]


def pick_body_cluster(clusters):
    """Largest cluster that is not dark (body paint); fall back to the largest if none qualifies."""
    for rgb, share in clusters:
        if max(rgb) >= DARK_V and share >= MIN_BODY_SHARE:
            return rgb, share
    return clusters[0]


def color_name(rgb):
    """Map an RGB color to a coarse name using HSV (OpenCV: H 0-179, S/V 0-255)."""
    h, s, v = cv2.cvtColor(np.uint8([[rgb]]), cv2.COLOR_RGB2HSV)[0, 0]
    h, s, v = int(h), int(s), int(v)
    if v < 50:
        return "black"
    if s < 40:
        return "white" if v > 200 else "silver" if v > 130 else "gray"
    if h < 10 or h >= 170:
        return "red"
    if h < 22:
        return "brown" if v < 130 else "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 130:
        return "blue"
    if h < 160:
        return "purple"
    return "pink"


# --- Detect and extract colors ---
img_bgr = cv2.imread(COLOR_TEST_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read {COLOR_TEST_PATH}")

res = trained_model.predict(img_bgr, imgsz=416, device=FORCE_DEVICE, conf=COLOR_CONF, verbose=False,)[0]

frame_sat = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[..., 1].mean()
if frame_sat < 15:
    print(f"WARNING: mean saturation {frame_sat:.1f} - frame looks grayscale/IR, color can't be recovered.")
color_src = brighten_frame(img_bgr)   # detection ran on the original; colors use the brightened copy

rows, crops, swatches = [], [], []
annotated = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
for b in res.boxes:
    if res.names[int(b.cls)] not in COLOR_CLASSES:
        continue
    box = b.xyxy[0].tolist()
    crop = crop_box_center(color_src, box)
    if crop.size == 0:
        continue
    clusters = color_clusters(crop)
    rgb, share = pick_body_cluster(clusters)
    name = color_name(rgb)
    car_id = f"car_{len(rows) + 1}"

    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), rgb, 3)
    cv2.putText(annotated, f"{car_id}: {name}", (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    crops.append((car_id, name, share, cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
    swatches.append(np.full((40, 80, 3), rgb, dtype=np.uint8))
    rows.append({
        "car": car_id,
        "conf": round(float(b.conf), 3),
        "box": [round(v, 1) for v in box],
        "rgb": rgb,
        "hex": "#%02x%02x%02x" % rgb,
        "color": name,
        "cluster_share": round(float(share), 2),
        "all_clusters": " ".join("#%02x%02x%02x:%.0f%%" % (*c, 100 * sh) for c, sh in clusters),
    })

print(f"{os.path.basename(COLOR_TEST_PATH)}: {len(rows)} car(s) detected")
if rows:
    print(pd.DataFrame(rows).to_string(index=False))

# --- Visualize ---
plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title("Detected cars with dominant color")
plt.show()

if rows:
    fig, axes = plt.subplots(2, len(rows), figsize=(3 * len(rows), 4), squeeze=False)
    for i, ((car_id, name, share, crop_rgb), sw) in enumerate(zip(crops, swatches)):
        axes[0, i].imshow(crop_rgb)
        axes[0, i].set_title(f"{car_id}: {name} ({share:.0%})", fontsize=9)
        axes[1, i].imshow(sw)
        for ax in (axes[0, i], axes[1, i]):
            ax.axis("off")
    plt.suptitle("Analysed crop (top) and dominant color (bottom)")
    plt.tight_layout()
    plt.show()

# Dynamic Spots

Using the model's prediction at every moment to detect occupation

## 14. Car permanence: arrives / stays / leaves

Given a series of photos from the same fixed camera, this section decides for each car whether it **arrives**, **stays** or **leaves**, and keeps a persistent ID (`#1`, `#2`, ...) so we know it is the *same* car across frames.

**How it works**

1. **Detect** every `Car` in each frame with `trained_model`.
2. **Describe** each detection with a CLIP image embedding (zero-shot, no training). This is the car's "appearance fingerprint", used instead of color.
3. **Match** each frame's detections to the cars already being tracked (Hungarian assignment). A pair is accepted only if **both** tests pass:
   - **position**: the boxes overlap enough (a parked car does not move), and
   - **appearance**: the embeddings are similar enough (a *different* car in the same spot fails this test).
4. **Decide**:
   - matched → **stays**
   - unmatched detection → **arrives** (new ID)
   - tracked car with no match → **leaves**

The frames are processed **in the order listed in `PERM_SEQUENCE`** (not sorted by filename). A repeated name is the same image, so its cars simply "stay". The first `EXPECTED_EMPTY_FRAMES` frames are known to be empty, which lets the cell warn about false positives.

### Tunable constants

| Constant | Default | What it controls | Raise it when... | Lower it when... |
|---|---|---|---|---|
| `PERM_CONF` | `0.25` | Minimum YOLO confidence for a box to count as a car. | Empty frames show phantom cars (false positives). | Real cars are being missed. |
| `IOU_THR` | `0.4` | Minimum box overlap (IoU) between a car's last known box and a new detection. | Different cars in neighbouring spots get merged. | The same parked car is called new because its box jitters between frames. |
| `SIM_THR` | `0.85` | Minimum CLIP cosine similarity (0 to 1) between a detection and the car's stored appearance. | A different car in the same spot is still treated as the same car. | The same car is split into "leaves" + "arrives" (e.g. day vs. night lighting). |
| `MISS_TOLERANCE` | `0` | Frames a car may go undetected before it counts as **leaving**. `0` = leaves on the first miss. | Detections flicker and cars "leave" and "arrive" spuriously (try `1` or `2`). | You need leaves reported immediately. |
| `REID_SIM_THR` | `0.92` | Only adds a note: a new arrival this similar to a car that already left is flagged "possibly #k again". It never reuses the ID. | The note fires for look-alike cars. | Returning cars are not being flagged. |
| `EXPECTED_EMPTY_FRAMES` | `2` | How many leading frames are known to be empty. Detections there trigger a warning. | The empty stretch at the start is longer. | There is no known-empty stretch (set `0`). |

**Tuning tip:** the cell prints a *match candidates* table with the real `iou` and `sim` of every pair it considered, and whether it matched. Set `IOU_THR` and `SIM_THR` between the values of pairs you know are the same car and pairs you know are different.

**Limitations**

- A car that leaves and later returns gets a **new ID**. Only the `REID_SIM_THR` note links them.
- CLIP is not a dedicated re-identification model, so look-alike cars (same model and color) can be confused, especially in dark frames.
- With `MISS_TOLERANCE > 0`, a car still unseen when the sequence ends is reported as "leaves" with the note *unconfirmed: sequence ended*.


In [ ]:
# --- 14. Car permanence: arrives / stays / leaves ---
# Requires from earlier sections: trained_model, FORCE_DEVICE, BASE_DIR
import open_clip
from scipy.optimize import linear_sum_assignment

# Frames in the order given (a repeated name is the same image, so the car simply "stays")
PERM_SEQUENCE = [
    "reference_image",
    "reference_image",
    "20260914191018",
    "20260914191018",
    "20260912171801",
    "20260912171801",
    "20260912191134",
    "20260912191134",
    "reference_image",
]
PERM_PATHS = [BASE_DIR + name + ".jpg" for name in PERM_SEQUENCE]

PERM_CONF = 0.25              # YOLO confidence
IOU_THR = 0.4                 # min box overlap with the car's last known position
SIM_THR = 0.65                # min CLIP cosine similarity to the car's stored appearance
MISS_TOLERANCE = 0            # frames a car may go undetected before it "leaves" (use 1-2 for flickery detections)
REID_SIM_THR = 0.92           # new arrival this similar to a car that left -> noted as possibly the same car
EXPECTED_EMPTY_FRAMES = 2     # the first N frames are known to be empty

# --- Appearance embedding (CLIP, zero-shot) ---
EMB_DEVICE = "cpu" if FORCE_DEVICE == "cpu" else f"cuda:{FORCE_DEVICE}"
if "clip_model" not in globals():
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    clip_model = clip_model.to(EMB_DEVICE).eval()


@torch.no_grad()
def embed_crops(crops_rgb):
    if not crops_rgb:
        return np.zeros((0, 512), dtype=np.float32)
    batch = torch.stack([clip_preprocess(Image.fromarray(c)) for c in crops_rgb]).to(EMB_DEVICE)
    e = clip_model.encode_image(batch).float()
    e = e / e.norm(dim=-1, keepdim=True)
    return e.cpu().numpy()


_PERM_CACHE = {}


def detect_cars(path):
    """-> (image_bgr, [{'box', 'conf', 'emb'}, ...]); cached per file."""
    if path in _PERM_CACHE:
        return _PERM_CACHE[path]
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    H, W = img.shape[:2]
    res = trained_model.predict(img, imgsz=416, device=FORCE_DEVICE, conf=PERM_CONF, verbose=False,iou=0.4)[0]
    dets, crops = [], []
    for b in res.boxes:
        if res.names[int(b.cls)] != "Car":
            continue
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        x1, y1, x2, y2 = max(0, int(x1)), max(0, int(y1)), min(W, int(x2)), min(H, int(y2))
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        dets.append({"box": (x1, y1, x2, y2), "conf": float(b.conf)})
        crops.append(cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    for d, e in zip(dets, embed_crops(crops)):
        d["emb"] = e
    _PERM_CACHE[path] = (img, dets)
    return _PERM_CACHE[path]


def box_iou_np(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def best_sim(embs, e):
    return float(np.max(np.stack(embs) @ e))


# --- Track cars across the sequence ---
labels = [os.path.basename(p)[:-4] for p in PERM_PATHS]
frames = [detect_cars(p) for p in PERM_PATHS]

tracks, events, match_log, frame_tracks = {}, [], {}, []
next_id = 1
for t, (img, dets) in enumerate(frames):
    alive = [i for i, tr in tracks.items() if tr["alive"]]
    score = np.zeros((len(alive), len(dets)))
    for a, tid in enumerate(alive):
        for j, d in enumerate(dets):
            iou = box_iou_np(tracks[tid]["box"], d["box"])
            sim = best_sim(tracks[tid]["embs"], d["emb"])
            if iou >= IOU_THR and sim >= SIM_THR:
                score[a, j] = iou * sim
            if iou > 0.1:
                match_log[(t, tid, j)] = {"frame": t, "image": labels[t], "car": tid, "det": j,
                                          "iou": round(iou, 2), "sim": round(sim, 3), "matched": False}
    matched = {}
    if score.size:
        for a, j in zip(*linear_sum_assignment(-score)):
            if score[a, j] > 0:
                matched[j] = alive[a]
                match_log[(t, alive[a], j)]["matched"] = True

    cur = []
    for j, d in enumerate(dets):
        if j in matched:
            tid = matched[j]
            tr = tracks[tid]
            note = f"re-detected after {tr['missed']} missed frame(s)" if tr["missed"] else ""
            tr.update(box=d["box"], missed=0, last=t)
            tr["embs"] = (tr["embs"] + [d["emb"]])[-5:]
            event = "stays"
        else:
            tid, next_id = next_id, next_id + 1
            note = ""
            dead = [(k, best_sim(v["embs"], d["emb"])) for k, v in tracks.items() if not v["alive"]]
            if dead:
                k, s = max(dead, key=lambda x: x[1])
                if s >= REID_SIM_THR:
                    note = f"possibly #{k} again (sim {s:.2f})"
            tracks[tid] = {"box": d["box"], "embs": [d["emb"]], "first": t, "last": t,
                           "missed": 0, "alive": True, "left": None}
            event = "arrives"
        events.append({"frame": t, "image": labels[t], "car": tid, "event": event, "note": note})
        cur.append((tid, d["box"]))
    frame_tracks.append(cur)

    for tid in alive:
        tr = tracks[tid]
        if tr["last"] == t:
            continue
        tr["missed"] += 1
        if tr["missed"] > MISS_TOLERANCE:
            tr["alive"], tr["left"] = False, tr["last"] + 1
            events.append({"frame": tr["left"], "image": labels[tr["left"]], "car": tid,
                           "event": "leaves", "note": ""})

for tid, tr in tracks.items():   # sequence ended while a car was still unseen (only if MISS_TOLERANCE > 0)
    if tr["alive"] and tr["missed"] > 0:
        tr["alive"], tr["left"] = False, tr["last"] + 1
        events.append({"frame": tr["left"], "image": labels[tr["left"]], "car": tid,
                       "event": "leaves", "note": "unconfirmed: sequence ended"})

# --- Report ---
early = [t for t in range(min(EXPECTED_EMPTY_FRAMES, len(frames))) if frames[t][1]]
if early:
    print(f"WARNING: frames {early} should be empty but have detections (false positives) - raise PERM_CONF.")

if not events:
    print("No cars detected in any frame.")
else:
    events_df = pd.DataFrame(events).sort_values(["frame", "car"]).reset_index(drop=True)
    print("EVENTS"); print(events_df.to_string(index=False))

    seen = events_df[events_df.event != "leaves"].groupby("car").size()
    summary = pd.DataFrame([{
        "car": f"#{tid}",
        "arrives_frame": tr["first"],
        "leaves_frame": tr["left"] if tr["left"] is not None else "-",
        "frames_seen": int(seen.get(tid, 0)),
    } for tid, tr in tracks.items()])
    print("\nPER CAR"); print(summary.to_string(index=False))

    if match_log:
        print("\nMATCH CANDIDATES (IoU > 0.1) - use iou/sim to tune IOU_THR and SIM_THR")
        print(pd.DataFrame(match_log.values()).to_string(index=False))

    # --- Visualize: persistent IDs across frames ---
    cmap = plt.get_cmap("tab10")
    cols = 3
    rows = (len(frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for t, ax in enumerate(axes.flat[:len(frames)]):
        vis = cv2.cvtColor(frames[t][0], cv2.COLOR_BGR2RGB).copy()
        fs = max(0.7, vis.shape[1] / 900)
        for tid, (x1, y1, x2, y2) in frame_tracks[t]:
            color = tuple(int(255 * c) for c in cmap((tid - 1) % 10)[:3])
            cv2.rectangle(vis, (x1, y1), (x2, y2), color, 3)
            cv2.putText(vis, f"#{tid}", (x1, max(24, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, color, max(2, int(fs * 2)))
        ev = ", ".join(f"{r.event} #{r.car}" for r in events_df[events_df.frame == t].itertuples())
        ax.imshow(vis)
        ax.set_title(f"{t}: {labels[t]}\n{ev or 'no events'}", fontsize=9)
    plt.tight_layout()
    plt.show()

## 15. Conclusions

### Results

- **Detection:** the fine-tuned YOLO model detects cars with sufficient precision for this task. It detects the **rear of a car noticeably better** than other views.
- **Model choice:** among the architectures compared in Section 7b, **YOLOv8m performed best** (highest validation mAP50). A possible reason is the relatively small size of the training dataset: it may not provide enough data for the other architectures to show an advantage.
- **Car presence:** the combination of YOLO detections, box overlap (IoU) and OpenCLIP appearance matching reliably confirms that a car is still present across captures (Section 14).
- **Spot matching:** parking spots can be matched with the detected cars (Sections 10 to 12).
- **Color:** K-Means color detection works well (Section 13).

### Dynamic Spots vs Static Spots

| | Static Spots (Section 10) | Dynamic Spots (Section 14) |
|---|---|---|
| **Idea** | Spots are calibrated once from a reference image and then reused. | Occupancy comes from the model's predictions at every moment. |
| **Strengths** | Consistency: the spots never change between frames. | Stronger car detection. No dependence on image-spot calibration. |
| **Weaknesses** | Adapts worse to different types of cars, since each spot is derived from the car that was parked there in the reference image. This may lower precision. | Spot control is harder: no spot is predefined, so every detection is a candidate "spot". |

Dynamic Spots need a **higher confidence threshold** to prevent other cars, or other objects in the background, from being detected as possible "spots". A further improvement would be to **force a fixed number of spots per camera**.

### What didn't work

- **Rotation:** rotating the images made the model less confident (Section 9b). The training dataset already contains cars in any position and orientation, so the model gains nothing from aligned input.
- **PKLot dataset** ([Kaggle](https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset)): it is much larger than the dataset used here, but its pictures show proper parking spaces with painted lines (usually yellow), and it labels *spots* instead of *cars*. The model became overly reliant on those markings and did not work properly when they were absent.
- **Motorcycles:** results were poor, because the dataset is focused on cars.
- **Oriented bounding boxes (OBB):** we tried YOLO26-OBB, trained on an OBB version of the current dataset. Model accuracy was lower, possibly because the task is harder for the model.

### Future work

- Further comparison of Dynamic Spots vs Static Spots.
- Tuning of the detection parameters (confidence, IoU and similarity thresholds).
- Combining predictions over several frames, to smooth out missed or flickering detections.
- Using IoU to handle overlapping boxes.